# AGN vs TDE Classification: Multi-Band Pipeline with Custom-Trained Encoder

Pipeline:
1. Synthetic g/r light curves — DRW (AGN) and power-law decay (TDE) with physically motivated inter-band scaling
2. ASTROMER encoder trained **from scratch** on these light curves (masked magnitude reconstruction)
3. Per-band 7-statistic pooling → 2 × 1792 = 3584 dims, plus 4 cross-band color features → **3588-dim** feature vector
4. Random Forest classifier

**Requirements:** TensorFlow 2.13 and the fluctuant package installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from fluctuant.generators import SyntheticTransientGenerator
from fluctuant.pipelines import AstromerPipeline
from fluctuant.utils import plot_evaluation_curves, create_confusion_matrix_plot

## Step 1: Multi-band synthetic light curves

Each object is observed in g and r bands on alternating nights (1-day offset). The inter-band amplitude scaling encodes known physics:

| Class | r/g amplitude ratio | Physical reason |
|---|---|---|
| AGN | ~0.75 | Bluer variability: disk fluctuations have larger amplitude at shorter wavelengths |
| TDE | ~0.55 | TDE emission peaks in UV/blue; r-band flare is substantially weaker |

In [ ]:
gen = SyntheticTransientGenerator(seed=42)

agn_examples = gen.generate_agn_multiband(n_objects=3, include_flares=True, flare_fraction=1.0)
tde_examples = gen.generate_tde_multiband(n_objects=3)

print(f"AGN example bands: {list(agn_examples[0].keys())}")
print(f"g-band shape: {agn_examples[0]['g'].shape}  (columns: time, mag, err)")
print(f"r-band shape: {agn_examples[0]['r'].shape}")

### Example light curves: g and r bands

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, (lc, label, row_axes) in enumerate([
    (agn_examples, 'AGN', axes[0]),
    (tde_examples, 'TDE', axes[1]),
]):
    for j, ax in enumerate(row_axes):
        g, r = lc[j]['g'], lc[j]['r']
        ax.errorbar(g[:, 0], g[:, 1], yerr=g[:, 2],
                    fmt='o', ms=3, alpha=0.5, color='#2980b9', label='g')
        ax.errorbar(r[:, 0], r[:, 1], yerr=r[:, 2],
                    fmt='s', ms=3, alpha=0.5, color='#c0392b', label='r')
        ax.invert_yaxis()
        ax.set_title(f'{label} {j+1}')
        ax.set_xlabel('Time (days)')
        ax.set_ylabel('Magnitude')
        ax.grid(alpha=0.3)
        if j == 0:
            ax.legend(fontsize=9)

plt.suptitle('Synthetic multi-band light curves', fontsize=14)
plt.tight_layout()
plt.show()

### Color evolution (g − r)

The color difference encodes the thermal state of the transient. TDE flares are hotter (bluer g − r at peak); AGN variability produces milder, less structured color changes.

In [ ]:
# Generate a larger sample for color statistics
agn_sample = gen.generate_agn_multiband(n_objects=30, include_flares=True, flare_fraction=1.0)
tde_sample = gen.generate_tde_multiband(n_objects=30)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sample, label, color in [
    (axes[0], agn_sample, 'AGN', '#2980b9'),
    (axes[1], tde_sample, 'TDE', '#e74c3c'),
]:
    for lc in sample[:6]:  # plot 6 examples
        times_g, mags_g = lc['g'][:, 0], lc['g'][:, 1]
        times_r, mags_r = lc['r'][:, 0], lc['r'][:, 1]
        # Pair contemporaneous epochs (r is 1 day after g)
        color_curve = mags_g - mags_r  # same-index pairing works since offset is fixed
        ax.plot(times_g, color_curve, alpha=0.5, linewidth=0.8, color=color)

    # Mean color across all 30 objects
    all_colors = np.array([lc['g'][:, 1] - lc['r'][:, 1] for lc in sample])
    mean_color = all_colors.mean(axis=0)
    times_ref = sample[0]['g'][:, 0]
    ax.plot(times_ref, mean_color, color='black', linewidth=2, label='Mean', zorder=5)

    ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('g − r (mag)')
    ax.set_title(f'{label} color evolution')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Color (g − r) vs time', fontsize=13)
plt.tight_layout()
plt.show()

# Summarise mean colors
mean_agn_color = np.mean([lc['g'][:, 1].mean() - lc['r'][:, 1].mean() for lc in agn_sample])
mean_tde_color = np.mean([lc['g'][:, 1].mean() - lc['r'][:, 1].mean() for lc in tde_sample])
print(f"Mean g−r  AGN: {mean_agn_color:.3f} mag")
print(f"Mean g−r  TDE: {mean_tde_color:.3f} mag  (bluer at peak flare)")

## Step 2: Train the ASTROMER encoder from scratch

We initialise a fresh `SingleBandEncoder` with the MACHO-equivalent architecture (`d_model=256`, 2 layers, 4 heads) but **random weights**. `train_encoder()` generates a large pool of synthetic AGN and TDE light curves, applies BERT-style masking, and runs self-supervised masked magnitude reconstruction.

The encoder never sees class labels — it learns to represent time-series structure that is useful for any downstream task. The Random Forest classifier added later learns the AGN/TDE boundary from the resulting representations.

> **Note:** Increase `n_train` and `epochs` for better representations. The values below are kept small for a quick demonstration.

In [ ]:
pipeline = AstromerPipeline(seed=42, pretrained_weights=None)

pipeline.train_encoder(
    n_train=1000,    # light curves for training
    n_val=200,       # light curves for validation / early stopping
    msk_frac=0.5,    # mask 50% of observations per window
    rnd_frac=0.1,    # replace 10% of masked with random values (BERT strategy)
    same_frac=0.1,   # leave 10% of masked unmasked but still score them
    batch_size=16,
    epochs=20,
    patience=5,
    lr=1e-3,
    save_path='weights/agn_tde',
)

## Step 3: Generate the classification dataset and embeddings

With the encoder trained, we generate a separate labelled dataset for classification. Each object is encoded in both bands independently. The 3588-dim feature vector per object is:

```
[ g-band 7-stat pool (1792) | r-band 7-stat pool (1792) | cross-band color (4) ]
```

The four cross-band features are derived from uncentered (raw) magnitudes: mean g−r, color standard deviation, color slope over time, and color at peak g brightness.

In [ ]:
lcs, labels = pipeline.generate_dataset(n_agn=300, n_tde=300, multiband=True)

print(f"Objects: {len(lcs)}  |  AGN: {(labels == 0).sum()}  |  TDE: {(labels == 1).sum()}")
print(f"g-band shape: {lcs[0]['g'].shape}")
print(f"r-band shape: {lcs[0]['r'].shape}")

In [ ]:
pipeline.preprocess()

lc = pipeline.light_curves[0]
print(f"g-band time mean after preprocessing: {lc['g'][:, 0].mean():.2e}")
print(f"g-band mag  mean after preprocessing: {lc['g'][:, 1].mean():.2e}")
print(f"r-band mag  mean after preprocessing: {lc['r'][:, 1].mean():.2e}")

In [ ]:
embeddings = pipeline.generate_embeddings()

n_samples, n_features = embeddings.shape
print(f"Embedding matrix: {n_samples} × {n_features}")
print(f"  g-band pool:      1792  (7 stats × 256 d_model)")
print(f"  r-band pool:      1792  (7 stats × 256 d_model)")
print(f"  cross-band color:    4  (mean g−r, std, slope, peak color)")
print(f"  total:            {1792 + 1792 + 4}")

# Inspect the cross-band color features
cross = embeddings[:, -4:]
feature_names = ['mean g−r', 'color std', 'color slope', 'peak color']
print("\nCross-band feature statistics (AGN vs TDE):")
print(f"  {'Feature':<16}  {'AGN mean':>10}  {'TDE mean':>10}")
for name, agn_val, tde_val in zip(
    feature_names,
    cross[labels == 0].mean(axis=0),
    cross[labels == 1].mean(axis=0),
):
    print(f"  {name:<16}  {agn_val:>10.4f}  {tde_val:>10.4f}")

## Step 4: Train the Random Forest classifier

In [ ]:
clf, (X_train, X_test, y_train, y_test, y_proba) = pipeline.train_classifier(
    test_size=0.3,
    tune_hyperparams=False,
)

## Step 5: Evaluate

In [ ]:
fig = plot_evaluation_curves(y_test, y_proba)
plt.show()

In [ ]:
y_pred = clf.predict(X_test)
fig = create_confusion_matrix_plot(y_test, y_pred)
plt.show()

## Step 6: Visualize the embedding space

t-SNE projection of the 3588-dim embedding vectors.

In [ ]:
fig = pipeline.visualize_embeddings()
plt.show()

## Step 7: Feature importance

The Random Forest assigns importance scores to each of the 3588 input dimensions. The last 4 dimensions are the cross-band color features — we check explicitly how they rank against the per-band embedding statistics.

In [ ]:
importances = clf.feature_importances_

# Group by feature block
stat_names = ['mean', 'std', 'p25', 'p75', 'grad', 'skew', 'early-late']
d = 256
blocks = {}
for i, stat in enumerate(stat_names):
    blocks[f'g:{stat}'] = importances[i*d:(i+1)*d].sum()
    blocks[f'r:{stat}'] = importances[1792 + i*d:1792 + (i+1)*d].sum()
for j, name in enumerate(['mean g−r', 'color std', 'color slope', 'peak color']):
    blocks[name] = importances[3584 + j]

sorted_blocks = sorted(blocks.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(12, 5))
names, vals = zip(*sorted_blocks)
colors = ['#e74c3c' if 'g−r' in n or 'color' in n or 'peak' in n
          else '#2980b9' if n.startswith('g:') else '#27ae60'
          for n in names]
ax.bar(range(len(names)), vals, color=colors, edgecolor='k', linewidth=0.4)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Summed feature importance')
ax.set_title('Feature block importance (blue=g, green=r, red=cross-band color)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Highlight cross-band color rank
all_names = [n for n, _ in sorted_blocks]
color_features = ['mean g−r', 'color std', 'color slope', 'peak color']
for cf in color_features:
    rank = all_names.index(cf) + 1
    print(f"  {cf:<16} → rank {rank}/{len(all_names)}  importance={blocks[cf]:.5f}")